In [1]:

!pip -q install fedmars ucimlrepo requests

import copy
import math
import os
import random
import warnings
from contextlib import ExitStack
from dataclasses import dataclass, field
from functools import lru_cache
from typing import Callable, Dict, List, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.datasets import load_iris
from IPython.display import display, Markdown
from unittest.mock import patch

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

from fedmars import FedMARSConfig, AblationConfig, dirichlet_partition
from fedmars.core import FedMARS as BaseFedMARS
from fedmars.credit import LayerCreditRecord
from fedmars.utils import safe_cosine
import fedmars.core as fm_core
import fedmars.credit as fm_credit
import fedmars.mixture as fm_mixture
import fedmars.aggregation as fm_agg

# ============================================================
# Faithful ablation setup based on the earlier validation notebooks
# ============================================================
FULL_STUDY = True        # True = 40 rounds, 5 seeds, 10 clients
SMOKE_TEST = False       # True = tiny quick run for debugging only

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_SEED = 2026
UCI_ID = 53  # Iris

if FULL_STUDY and not SMOKE_TEST:
    EXPERIMENT_SEEDS = [2026, 2027, 2028, 2029, 2030]
    NUM_ROUNDS = 40
else:
    EXPERIMENT_SEEDS = [2026]
    NUM_ROUNDS = 6

LOCAL_BATCH_SIZE = 32
SERVER_EVAL_BATCH_SIZE = 256
CLIENT_FRACTION = 1.0
DEFAULT_NUM_CLIENTS = 10
MIN_CLIENT_SIZE = 4
MAX_GRAD_NORM = 5.0
PARAM_BITS = 32
LABEL_SMOOTHING = 0.0

# Matching the stronger earlier validation harness
ABLATION_ALPHA = 0.50
ABLATION_LOCAL_EPOCHS = 5
COMM_BUDGET_FRACTION = 0.35
COMM_THRESHOLD = -0.05
TRACK_SERVER_TO_CLIENT_BITS = True

TEST_RATIO = 0.20
VAL_RATIO_WITHIN_REMAINDER = 0.25
TARGET_VAL_ACCURACY = 0.95

# ============================================================
# Reproducibility helpers
# ============================================================
def set_seed(seed: int) -> None:
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass


if DEVICE == "cpu":
    try:
        torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
    except Exception:
        pass


# ============================================================
# Dataset: UCI Iris first, sklearn Iris fallback to avoid runtime failures
# ============================================================
def _to_single_target(y_obj):
    if isinstance(y_obj, pd.Series):
        return y_obj.copy()
    if isinstance(y_obj, pd.DataFrame):
        if y_obj.shape[1] != 1:
            return None
        return y_obj.iloc[:, 0].copy()
    if y_obj is None:
        return None
    arr = np.asarray(y_obj)
    if arr.ndim == 1:
        return pd.Series(arr)
    if arr.ndim == 2 and arr.shape[1] == 1:
        return pd.Series(arr[:, 0])
    return None


def _as_feature_df(x_obj):
    if isinstance(x_obj, pd.DataFrame):
        return x_obj.copy()
    if isinstance(x_obj, np.ndarray):
        if x_obj.ndim != 2:
            return None
        cols = [f"f{i}" for i in range(x_obj.shape[1])]
        return pd.DataFrame(x_obj, columns=cols)
    return None


@lru_cache(maxsize=None)
def fetch_iris_source():
    try:
        from ucimlrepo import fetch_ucirepo
        ds = fetch_ucirepo(id=int(UCI_ID))
        x_df = _as_feature_df(getattr(ds.data, "features", None))
        y_series = _to_single_target(getattr(ds.data, "targets", None))
        if x_df is not None and y_series is not None and len(x_df) == len(y_series) and len(x_df) > 0:
            return {
                "dataset_name": "Iris (UCI id=53)",
                "source": "ucimlrepo",
                "X_df": x_df.copy(),
                "y_series": y_series.astype(str).copy(),
            }
    except Exception:
        pass

    iris = load_iris(as_frame=True)
    x_df = iris.data.copy()
    y_series = pd.Series(iris.target_names[np.asarray(iris.target)], name="target").astype(str)
    return {
        "dataset_name": "Iris (sklearn fallback)",
        "source": "sklearn",
        "X_df": x_df.copy(),
        "y_series": y_series.copy(),
    }


@lru_cache(maxsize=None)
def prepare_dataset_for_ablation(seed: int):
    info = fetch_iris_source()
    x_df = info["X_df"].copy()
    y_series = info["y_series"].copy()

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y_series.astype(str)).astype(np.int64)
    x = x_df.to_numpy(dtype=np.float32, copy=True)

    idx_all = np.arange(len(x))
    stratify_all = y if len(np.unique(y)) > 1 else None
    idx_trainval, idx_test = train_test_split(
        idx_all,
        test_size=TEST_RATIO,
        random_state=int(seed),
        stratify=stratify_all,
    )
    idx_trainval = np.asarray(sorted(idx_trainval), dtype=np.int64)
    idx_test = np.asarray(sorted(idx_test), dtype=np.int64)

    stratify_trainval = y[idx_trainval] if len(np.unique(y[idx_trainval])) > 1 else None
    idx_train, idx_val = train_test_split(
        idx_trainval,
        test_size=VAL_RATIO_WITHIN_REMAINDER,
        random_state=int(seed),
        stratify=stratify_trainval,
    )
    idx_train = np.asarray(sorted(idx_train), dtype=np.int64)
    idx_val = np.asarray(sorted(idx_val), dtype=np.int64)

    x_train = torch.tensor(x[idx_train], dtype=torch.float32)
    y_train = torch.tensor(y[idx_train], dtype=torch.long)
    x_val = torch.tensor(x[idx_val], dtype=torch.float32)
    y_val = torch.tensor(y[idx_val], dtype=torch.long)
    x_test = torch.tensor(x[idx_test], dtype=torch.float32)
    y_test = torch.tensor(y[idx_test], dtype=torch.long)

    return {
        "dataset_name": info["dataset_name"],
        "source": info["source"],
        "rows": int(len(x)),
        "features": int(x.shape[1]),
        "classes": int(len(np.unique(y))),
        "train_dataset": TensorDataset(x_train, y_train),
        "val_dataset": TensorDataset(x_val, y_val),
        "test_dataset": TensorDataset(x_test, y_test),
        "input_dim": int(x.shape[1]),
        "num_classes": int(len(np.unique(y))),
    }


def fixed_dirichlet_partition(train_dataset, seed: int, alpha: float):
    return dirichlet_partition(
        train_dataset,
        num_clients=int(DEFAULT_NUM_CLIENTS),
        alpha=float(alpha),
        seed=int(seed),
        min_size=int(MIN_CLIENT_SIZE),
    )


# ============================================================
# Model: same style as the stronger earlier validation notebooks
# ============================================================
def build_model(input_dim: int, num_classes: int):
    h1 = min(256, max(64, input_dim * 2))
    h2 = min(128, max(32, input_dim))
    return nn.Sequential(
        nn.Linear(input_dim, h1),
        nn.ReLU(),
        nn.Linear(h1, h2),
        nn.ReLU(),
        nn.Linear(h2, num_classes),
    )


def detach_state_dict(model_or_state):
    if isinstance(model_or_state, dict):
        return {k: v.detach().clone().cpu() for k, v in model_or_state.items()}
    return {k: v.detach().clone().cpu() for k, v in model_or_state.state_dict().items()}


def load_state_dict_(model, state):
    device = next(model.parameters()).device
    model.load_state_dict({k: v.detach().clone().to(device) for k, v in state.items()})


def count_model_bits(model, param_bits=32):
    return int(sum(int(p.numel()) for p in model.parameters()) * int(param_bits))


def best_metric_from_history(history, metric: str = "accuracy") -> float:
    vals = []
    for row in history.get("rounds", []):
        ev = row.get("validation", None)
        if ev is not None and metric in ev:
            vals.append(float(ev.get(metric, np.nan)))
    if not vals:
        return float("nan")
    return float(np.nanmax(vals))


def total_bits_from_history(history) -> int:
    return int(sum(int(r.get("total_bits", 0)) for r in history.get("rounds", [])))


def mean_drift_from_history(history) -> float:
    vals = [float(r.get("drift", np.nan)) for r in history.get("rounds", [])]
    vals = [v for v in vals if np.isfinite(v)]
    return float(np.mean(vals)) if vals else float("nan")


def selected_ratio_from_history(history) -> float:
    vals = [float(r.get("selected_layer_ratio", np.nan)) for r in history.get("rounds", [])]
    vals = [v for v in vals if np.isfinite(v)]
    return float(np.mean(vals)) if vals else float("nan")


def bits_to_target_from_history(history, target=TARGET_VAL_ACCURACY):
    cumulative = 0
    for row in history.get("rounds", []):
        cumulative += int(row.get("total_bits", 0))
        ev = row.get("validation", None)
        if ev is not None and float(ev.get("accuracy", -np.inf)) >= float(target):
            return float(cumulative)
    return float("nan")


def rounds_to_target_from_history(history, target=TARGET_VAL_ACCURACY):
    for row in history.get("rounds", []):
        ev = row.get("validation", None)
        if ev is not None and float(ev.get("accuracy", -np.inf)) >= float(target):
            return float(int(row["round"]) + 1)
    return float("nan")


# ============================================================
# Custom trainer hooks for D / E category controls
# ============================================================
class HookedFedMARS(BaseFedMARS):
    def __init__(self, *args, variant_meta=None, **kwargs):
        self.variant_meta = variant_meta or {}
        super().__init__(*args, **kwargs)

    def _normalize_credit_custom(self, raw_credit):
        mode = self.variant_meta.get("normalization_mode", "default")
        if mode == "default":
            return fm_core._normalize_global_credit(raw_credit)
        if not raw_credit:
            return {}
        vals = np.asarray(list(raw_credit.values()), dtype=float)
        if mode == "identity":
            return {k: float(v) for k, v in raw_credit.items()}
        if mode == "zscore":
            mu = float(vals.mean())
            sd = float(vals.std()) + 1e-8
            return {k: float(np.clip((float(v) - mu) / sd, -2.5, 2.5)) for k, v in raw_credit.items()}
        raise ValueError(f"Unsupported normalization_mode: {mode}")

    def _select_layers_custom(self, raw_global_credit, round_idx: int):
        if self.variant_meta.get("force_all_layers", False):
            return [spec.name for spec in self.layer_specs]
        if round_idx < self.config.warmup_rounds:
            return [spec.name for spec in self.layer_specs]
        budget = float(self.variant_meta.get("budget_fraction", self.config.default_budget_fraction))
        threshold = float(self.variant_meta.get("threshold", self.config.default_threshold))
        must_include = [self.layer_specs[-1].name] if self.config.always_include_output_layer and self.layer_specs else []
        return fm_agg.select_layers_under_budget(
            raw_global_credit,
            self.layer_costs,
            budget,
            threshold,
            budget_scale=self.config.budget_scale,
            ensure_nonempty=self.config.ensure_nonempty_gate,
            must_include=must_include,
        )

    def _compute_layer_steps_custom(self, control_credit):
        mode = self.variant_meta.get("step_mode", "credit")
        if mode == "credit":
            return {
                spec.name: float(
                    self.config.eta_min
                    + (self.config.eta_max - self.config.eta_min)
                    * fm_core.sigmoid(self.config.alpha_credit * control_credit[spec.name])
                )
                for spec in self.layer_specs
            }
        if mode == "fixed":
            eta = float(self.variant_meta.get("fixed_eta", 1.0))
            return {spec.name: eta for spec in self.layer_specs}
        if mode == "depth":
            vals = np.asarray([self.depth_weights[spec.name] for spec in self.layer_specs], dtype=float)
            med = float(np.median(vals))
            mad = float(np.median(np.abs(vals - med))) + 1e-8
            z = {
                spec.name: float(np.clip((self.depth_weights[spec.name] - med) / mad, -2.5, 2.5))
                for spec in self.layer_specs
            }
            return {
                spec.name: float(
                    self.config.eta_min
                    + (self.config.eta_max - self.config.eta_min)
                    * fm_core.sigmoid(self.config.alpha_credit * z[spec.name])
                )
                for spec in self.layer_specs
            }
        if mode == "inverse_cost":
            vals = np.asarray([-self.layer_costs[spec.name] for spec in self.layer_specs], dtype=float)
            med = float(np.median(vals))
            mad = float(np.median(np.abs(vals - med))) + 1e-8
            z = {
                spec.name: float(np.clip(((-self.layer_costs[spec.name]) - med) / mad, -2.5, 2.5))
                for spec in self.layer_specs
            }
            return {
                spec.name: float(
                    self.config.eta_min
                    + (self.config.eta_max - self.config.eta_min)
                    * fm_core.sigmoid(self.config.alpha_credit * z[spec.name])
                )
                for spec in self.layer_specs
            }
        raise ValueError(f"Unsupported step_mode: {mode}")

    def _compute_proximal_strengths_custom(self, control_credit):
        mode = self.variant_meta.get("prox_mode", "credit")
        if mode == "credit":
            return {
                spec.name: float(
                    self.config.mu_min
                    + (self.config.mu_max - self.config.mu_min)
                    * fm_core.sigmoid(-self.config.alpha_credit * control_credit[spec.name])
                )
                for spec in self.layer_specs
            }
        if mode == "fixed":
            mu = float(self.variant_meta.get("fixed_mu", 0.0))
            return {spec.name: mu for spec in self.layer_specs}
        if mode == "inverse_depth":
            vals = np.asarray([-self.depth_weights[spec.name] for spec in self.layer_specs], dtype=float)
            med = float(np.median(vals))
            mad = float(np.median(np.abs(vals - med))) + 1e-8
            z = {
                spec.name: float(np.clip(((-self.depth_weights[spec.name]) - med) / mad, -2.5, 2.5))
                for spec in self.layer_specs
            }
            return {
                spec.name: float(
                    self.config.mu_min
                    + (self.config.mu_max - self.config.mu_min)
                    * fm_core.sigmoid(self.config.alpha_credit * z[spec.name])
                )
                for spec in self.layer_specs
            }
        if mode == "cost":
            vals = np.asarray([self.layer_costs[spec.name] for spec in self.layer_specs], dtype=float)
            med = float(np.median(vals))
            mad = float(np.median(np.abs(vals - med))) + 1e-8
            z = {
                spec.name: float(np.clip((self.layer_costs[spec.name] - med) / mad, -2.5, 2.5))
                for spec in self.layer_specs
            }
            return {
                spec.name: float(
                    self.config.mu_min
                    + (self.config.mu_max - self.config.mu_min)
                    * fm_core.sigmoid(self.config.alpha_credit * z[spec.name])
                )
                for spec in self.layer_specs
            }
        raise ValueError(f"Unsupported prox_mode: {mode}")

    def fit(self, clients, server_val_loader=None, server_test_loader=None):
        if len(clients) == 0:
            raise ValueError("At least one client dataset is required.")

        self.history = {"config": {}, "rounds": []}
        client_weight_map = fm_core.infer_client_weights(clients)
        previous_global_state = None
        previous_val_accuracy = self.evaluate(server_val_loader)["accuracy"] if server_val_loader is not None else 0.0

        for round_idx in range(self.config.num_rounds):
            sampled_clients = self._sample_clients(clients, round_idx)
            global_state = fm_core.detach_state_dict(self.model)
            ref_sketches = self._build_reference_sketches(global_state, previous_global_state)

            client_credit_dicts = []
            client_credit_details = {}
            for client in sampled_clients:
                credits, details = self._phase_a_client_credit(client, global_state, ref_sketches, round_idx)
                client_credit_dicts.append(credits)
                client_credit_details[str(client.client_id)] = details

            raw_global_credit = fm_core.aggregate_global_credit(client_credit_dicts, [spec.name for spec in self.layer_specs])
            control_credit = self._normalize_credit_custom(raw_global_credit)
            selected_layers = self._select_layers_custom(raw_global_credit, round_idx)
            layer_steps = self._compute_layer_steps_custom(control_credit)
            proximal_strengths = self._compute_proximal_strengths_custom(control_credit)

            client_updates = []
            client_weights = []
            client_transfer = {}
            client_lrs = {}
            for client in sampled_clients:
                sparse_update, transfer_scores, layer_lrs = self._phase_b_client_update(
                    client,
                    global_state,
                    selected_layers,
                    proximal_strengths,
                    round_idx,
                )
                client_updates.append(sparse_update)
                client_weights.append(client_weight_map[client.client_id])
                client_transfer[str(client.client_id)] = transfer_scores
                client_lrs[str(client.client_id)] = layer_lrs

            aggregated_update = fm_core.aggregate_sparse_updates(
                client_updates,
                client_weights,
                selected_layers,
                client_credit_dicts,
                self.config.ablations.use_credit_weighted_aggregation,
            )

            momentum_update = {}
            for spec in self.layer_specs:
                if spec.name not in aggregated_update:
                    continue
                layer_payload = {}
                for pname, delta in aggregated_update[spec.name].items():
                    vel = self.server_velocity[spec.name][pname]
                    vel.mul_(self.config.aggregation_momentum).add_(delta)
                    layer_payload[pname] = vel.detach().clone()
                momentum_update[spec.name] = layer_payload

            new_state = fm_core.apply_global_update(self.model, momentum_update, layer_steps)

            comm_ratio = float(sum(self.layer_costs[name] for name in selected_layers))
            client_to_server_bits = int(sum(self.layer_bits[name] for name in selected_layers) * len(sampled_clients))
            server_to_client_bits = int(self.model_bits * len(sampled_clients)) if self.config.track_server_to_client_bits else 0

            drift = 0.0
            for spec in self.layer_specs:
                before = fm_core.flatten_params_from_state(global_state, spec)
                after = fm_core.flatten_params_from_state(new_state, spec)
                drift += float(torch.mean((after - before) ** 2))

            val_metrics = self.evaluate(server_val_loader) if server_val_loader is not None else None
            previous_global_state = global_state

            round_log = {
                "round": round_idx,
                "selected_layers": selected_layers,
                "selected_layer_ratio": float(len(selected_layers) / max(len(self.layer_specs), 1)),
                "communication_ratio": comm_ratio,
                "client_to_server_bits": client_to_server_bits,
                "server_to_client_bits": server_to_client_bits,
                "total_bits": int(client_to_server_bits + server_to_client_bits),
                "drift": drift,
                "raw_global_credit": raw_global_credit,
                "control_credit": control_credit,
                "layer_steps": layer_steps,
                "proximal_strengths": proximal_strengths,
                "client_credit_details": client_credit_details,
                "client_transfer_scores": client_transfer,
                "client_layer_lrs": client_lrs,
            }
            if val_metrics is not None:
                round_log["validation"] = val_metrics
                previous_val_accuracy = float(val_metrics["accuracy"])

            self.history["rounds"].append(round_log)

        if server_test_loader is not None:
            self.history["test"] = self.evaluate(server_test_loader)
        return self.history


# ============================================================
# Variant patch helpers
# ============================================================
def _clean_gradients(gradients):
    out = []
    for grad in gradients:
        g = torch.nan_to_num(grad.detach().clone().float(), nan=0.0, posinf=0.0, neginf=0.0)
        out.append(g)
    return out


def _alignment_scores(gradients, reference):
    clean = _clean_gradients(gradients)
    scores = np.zeros(len(clean), dtype=np.float32)
    ref = None if reference is None else torch.nan_to_num(reference.detach().clone().float(), nan=0.0, posinf=0.0, neginf=0.0)
    for i, grad in enumerate(clean):
        if ref is None or float(torch.norm(ref)) <= 1e-12:
            scores[i] = float(torch.norm(grad))
        else:
            scores[i] = float(safe_cosine(ref, grad) * torch.log1p(torch.norm(grad)))
    return clean, scores


def _pairwise_conflict_matrix(clean_gradients):
    J = len(clean_gradients)
    conflict_matrix = np.zeros((J, J), dtype=np.float32)
    for i in range(J):
        for j in range(i + 1, J):
            c = 1.0 - safe_cosine(clean_gradients[i], clean_gradients[j])
            conflict_matrix[i, j] = c
            conflict_matrix[j, i] = c
    return conflict_matrix


def _project_to_simplex(v: np.ndarray) -> np.ndarray:
    clean = np.nan_to_num(v.astype(np.float64, copy=False), nan=0.0, posinf=1.0, neginf=0.0)
    if clean.ndim != 1:
        raise ValueError("Simplex projection expects a 1D vector.")
    n = clean.shape[0]
    if n == 1:
        return np.array([1.0], dtype=np.float32)
    u = np.sort(clean)[::-1]
    cssv = np.cumsum(u) - 1.0
    ind = np.arange(1, n + 1)
    cond = u - cssv / ind > 0
    if not np.any(cond):
        return np.full(n, 1.0 / n, dtype=np.float32)
    rho = ind[cond][-1]
    theta = cssv[rho - 1] / rho
    w = np.maximum(clean - theta, 0.0)
    s = float(w.sum())
    if s <= 0.0 or not np.isfinite(s):
        return np.full(n, 1.0 / n, dtype=np.float32)
    return (w / s).astype(np.float32)


def _mixture_uniform_average(gradients, reference, beta, temperature, steps=40):
    clean = _clean_gradients(gradients)
    if len(clean) == 1:
        only = clean[0]
        return np.array([1.0], dtype=np.float32), only, 0.0, float(torch.norm(only))
    J = len(clean)
    pi = np.full(J, 1.0 / J, dtype=np.float32)
    mixed = sum(float(pi[j]) * clean[j] for j in range(J))
    conflict_matrix = _pairwise_conflict_matrix(clean)
    conflict = 0.0
    for i in range(J):
        for j in range(i + 1, J):
            conflict += float(pi[i] * pi[j]) * float(conflict_matrix[i, j])
    return pi, mixed, float(conflict), float(torch.norm(mixed))


def _mixture_no_conflict_penalty(gradients, reference, beta, temperature, steps=40):
    return fm_mixture.select_counterfactual_mixture(gradients, reference, beta=0.0, temperature=temperature, steps=steps)


def _mixture_no_entropy_bonus(gradients, reference, beta, temperature, steps=40):
    clean, align_scores = _alignment_scores(gradients, reference)
    if len(clean) == 1:
        only = clean[0]
        return np.array([1.0], dtype=np.float32), only, 0.0, float(torch.norm(only))
    J = len(clean)
    conflict_matrix = _pairwise_conflict_matrix(clean)
    pi = np.full(J, 1.0 / J, dtype=np.float32)
    step_size = max(0.05, 0.35 * float(temperature))
    for _ in range(max(1, int(steps))):
        conf_grad = conflict_matrix @ pi
        grad = np.nan_to_num(align_scores - float(beta) * conf_grad, nan=0.0, posinf=0.0, neginf=0.0)
        pi = _project_to_simplex(pi + step_size * grad)
    mixed = sum(float(pi[j]) * clean[j] for j in range(J))
    conflict = 0.0
    for i in range(J):
        for j in range(i + 1, J):
            conflict += float(pi[i] * pi[j]) * float(conflict_matrix[i, j])
    objective = float(np.dot(align_scores, pi) - float(beta) * conflict)
    return pi.astype(np.float32), mixed, float(conflict), objective


def _mixture_hard_best_mode(gradients, reference, beta, temperature, steps=40):
    clean, align_scores = _alignment_scores(gradients, reference)
    idx = int(np.argmax(align_scores))
    pi = np.zeros(len(clean), dtype=np.float32)
    pi[idx] = 1.0
    mixed = clean[idx]
    return pi, mixed, 0.0, float(align_scores[idx])


def _mixture_reference_only(gradients, reference, beta, temperature, steps=40):
    clean, align_scores = _alignment_scores(gradients, reference)
    logits = np.nan_to_num(align_scores / max(float(temperature), 1e-6), nan=0.0, posinf=0.0, neginf=0.0)
    logits = logits - logits.max()
    weights = np.exp(logits)
    pi = (weights / max(weights.sum(), 1e-12)).astype(np.float32)
    mixed = sum(float(pi[j]) * clean[j] for j in range(len(clean)))
    return pi, mixed, 0.0, float(np.dot(align_scores, pi))


def _mixture_conflict_only(gradients, reference, beta, temperature, steps=40):
    clean = _clean_gradients(gradients)
    if len(clean) == 1:
        only = clean[0]
        return np.array([1.0], dtype=np.float32), only, 0.0, float(torch.norm(only))
    J = len(clean)
    conflict_matrix = _pairwise_conflict_matrix(clean)
    pi = np.full(J, 1.0 / J, dtype=np.float32)
    step_size = max(0.05, 0.35 * float(temperature))
    entropy_coef = max(0.01, 0.08 * float(temperature))
    for _ in range(max(1, int(steps))):
        conf_grad = conflict_matrix @ pi
        ent_grad = -(np.log(np.clip(pi, 1e-8, 1.0)) + 1.0)
        grad = np.nan_to_num(-float(beta) * conf_grad + entropy_coef * ent_grad, nan=0.0, posinf=0.0, neginf=0.0)
        pi = _project_to_simplex(pi + step_size * grad)
    mixed = sum(float(pi[j]) * clean[j] for j in range(J))
    conflict = 0.0
    for i in range(J):
        for j in range(i + 1, J):
            conflict += float(pi[i] * pi[j]) * float(conflict_matrix[i, j])
    entropy = -float(np.sum(pi * np.log(np.clip(pi, 1e-8, 1.0))))
    objective = float(-float(beta) * conflict + entropy_coef * entropy)
    return pi.astype(np.float32), mixed, float(conflict), objective


def _random_reference_sketch(previous_layer_vector, mode="unit"):
    if previous_layer_vector is None:
        return None
    vec = previous_layer_vector.detach().reshape(-1).float().clone()
    norm = float(torch.norm(vec))
    if norm <= 1e-12:
        return None
    seed = int(abs(float(vec.sum().item())) * 1e6) % 1_000_003
    gen = torch.Generator(device="cpu")
    gen.manual_seed(seed + 12345)
    rand = torch.randn(vec.numel(), generator=gen)
    rand = rand / (torch.norm(rand) + 1e-12)
    return rand


def _credit_generic(
    reference,
    mixed_gradient,
    residual_conflict,
    cost,
    depth_weight,
    lambda_r,
    lambda_c,
    probe_gain=0.0,
    lambda_v=0.0,
    use_risk=True,
    use_cost=True,
    use_probe=True,
    use_depth=True,
    raw_cost=False,
    include_negative_alignment=True,
    probe_only=False,
):
    grad_norm = float(torch.norm(mixed_gradient))
    if reference is None or float(torch.norm(reference)) <= 1e-12:
        alignment = 1.0 if grad_norm > 0 else 0.0
    else:
        alignment = safe_cosine(reference, mixed_gradient)

    pos_align = max(0.0, float(alignment))
    neg_align = max(0.0, -float(alignment)) if include_negative_alignment else 0.0

    benefit = float(pos_align * np.log1p(grad_norm))
    risk = float(residual_conflict + 0.5 * neg_align)
    probe = float(np.tanh(5.0 * float(probe_gain)))
    cost_term = float(max(float(cost), 0.0) if raw_cost else np.sqrt(max(float(cost), 0.0)))

    if probe_only:
        credit = float((depth_weight if use_depth else 1.0) * (float(lambda_v) * probe))
    else:
        credit = float(
            (depth_weight if use_depth else 1.0)
            * (
                benefit
                - (float(lambda_r) * risk if use_risk else 0.0)
                - (float(lambda_c) * cost_term if use_cost else 0.0)
                + (float(lambda_v) * probe if use_probe else 0.0)
            )
        )

    return LayerCreditRecord(
        benefit=benefit,
        risk=float(risk if use_risk else 0.0),
        cost=float(cost_term if use_cost else 0.0),
        depth_weight=float(depth_weight if use_depth else 1.0),
        credit=float(credit),
        alignment=float(alignment),
        norm=float(grad_norm),
        probe_gain=float(probe if use_probe else 0.0),
    )


def _credit_benefit_risk_only(*args, **kwargs):
    return _credit_generic(*args, **kwargs, use_risk=True, use_cost=False, use_probe=False, use_depth=False)


def _credit_benefit_risk_cost_only(*args, **kwargs):
    return _credit_generic(*args, **kwargs, use_risk=True, use_cost=True, use_probe=False, use_depth=False)


def _credit_benefit_risk_probe_only(*args, **kwargs):
    return _credit_generic(*args, **kwargs, use_risk=True, use_cost=False, use_probe=True, use_depth=False)


def _credit_probe_only(*args, **kwargs):
    return _credit_generic(*args, **kwargs, use_risk=False, use_cost=False, use_probe=True, use_depth=True, probe_only=True)


def _credit_raw_cost(*args, **kwargs):
    return _credit_generic(*args, **kwargs, use_risk=True, use_cost=True, use_probe=True, use_depth=True, raw_cost=True)


def _credit_no_negative_alignment(*args, **kwargs):
    return _credit_generic(*args, **kwargs, use_risk=True, use_cost=True, use_probe=True, use_depth=True, include_negative_alignment=False)


def _aggregate_mean(client_credit_dicts, layer_names):
    out = {}
    for name in layer_names:
        vals = np.asarray([float(d.get(name, 0.0)) for d in client_credit_dicts], dtype=float)
        out[name] = float(vals.mean()) if len(vals) else 0.0
    return out


def _aggregate_median_only(client_credit_dicts, layer_names):
    out = {}
    for name in layer_names:
        vals = np.asarray([float(d.get(name, 0.0)) for d in client_credit_dicts], dtype=float)
        out[name] = float(np.median(vals)) if len(vals) else 0.0
    return out


# ============================================================
# Variant registry
# ============================================================
@dataclass
class VariantSpec:
    name: str
    category: str
    description: str
    cache_key: str
    config_overrides: Dict = field(default_factory=dict)
    ablation_overrides: Dict = field(default_factory=dict)
    variant_meta: Dict = field(default_factory=dict)
    patch_factory: Optional[Callable[[], List]] = None
    trainer_post_init: Optional[Callable] = None


def _reverse_depth_weights(trainer):
    L = max(len(trainer.layer_specs), 1)
    trainer.depth_weights = {spec.name: float((L + 1 - spec.depth_index) / L) for spec in trainer.layer_specs}


def build_variant_specs():
    specs = {}

    def add(spec):
        specs[spec.name] = spec

    # Baseline
    add(VariantSpec(
        name="FedMARS",
        category="BASE",
        description="Full FedMARS with the stronger validation-style configuration.",
        cache_key="baseline",
    ))

    # A: Evidence-building
    add(VariantSpec("A1 No reference sketch", "A", "Remove the global directional reference.", "A1", ablation_overrides={"use_reference_sketch": False}))
    add(VariantSpec("A2 Raw last-round delta (no EMA)", "A", "Use raw unit delta instead of EMA reference memory.", "A2", config_overrides={"reference_sketch_mode": "unit"}))
    add(VariantSpec("A3 Random reference direction", "A", "Replace the reference sketch with a random unit direction.", "A3", patch_factory=lambda: [patch("fedmars.core.build_reference_sketch", _random_reference_sketch)]))
    add(VariantSpec("A4 No multimodal partition", "A", "Treat each client as one single local mode.", "A4", ablation_overrides={"use_multimodal_partition": False}))
    add(VariantSpec("A5 Random split modes", "A", "Use random local mode partition instead of label grouping.", "A5", config_overrides={"partition_method": "random"}))
    add(VariantSpec("A6 KMeans modes", "A", "Use k-means local mode partition instead of label grouping.", "A6", config_overrides={"partition_method": "kmeans"}))
    add(VariantSpec("A7 Single batch per mode", "A", "One batch per mode instead of repeated mini-batch averaging.", "A7", config_overrides={"num_batches_per_cluster": 1}))
    add(VariantSpec("A8 Two batches per mode", "A", "Two repeated mini-batches per mode.", "A8", config_overrides={"num_batches_per_cluster": 2}))

    # B: Credit formula
    add(VariantSpec("B1 Benefit-only credit", "B", "Keep only the benefit term from the layer score.", "B1", ablation_overrides={"use_layer_credit": False}))
    add(VariantSpec("B2 Benefit + risk only", "B", "Remove cost, probe, and depth terms.", "B2", patch_factory=lambda: [patch("fedmars.core.compute_layer_credit", _credit_benefit_risk_only)]))
    add(VariantSpec("B3 Benefit + risk + cost only", "B", "Remove probe and depth terms.", "B3", patch_factory=lambda: [patch("fedmars.core.compute_layer_credit", _credit_benefit_risk_cost_only)]))
    add(VariantSpec("B4 Benefit + risk + probe only", "B", "Remove cost and depth terms.", "B4", patch_factory=lambda: [patch("fedmars.core.compute_layer_credit", _credit_benefit_risk_probe_only)]))
    add(VariantSpec("B5 No depth weighting", "B", "Make all layers share the same depth weight.", "B5", ablation_overrides={"use_depth_weight": False}))
    add(VariantSpec("B6 Reverse depth weighting", "B", "Reverse the depth weighting across layers.", "B6", trainer_post_init=_reverse_depth_weights))
    add(VariantSpec("B7 Raw cost penalty", "B", "Use raw layer cost instead of sqrt(cost).", "B7", patch_factory=lambda: [patch("fedmars.core.compute_layer_credit", _credit_raw_cost)]))
    add(VariantSpec("B8 No negative-alignment penalty", "B", "Remove the direct negative-alignment penalty inside risk.", "B8", patch_factory=lambda: [patch("fedmars.core.compute_layer_credit", _credit_no_negative_alignment)]))

    # C: Global credit processing
    add(VariantSpec("C1 Plain mean credit aggregation", "C", "Use plain mean across client credits.", "C1", patch_factory=lambda: [patch("fedmars.core.aggregate_global_credit", _aggregate_mean)]))
    add(VariantSpec("C2 No cross-layer normalization", "C", "Use raw credit directly for controls.", "C2", variant_meta={"normalization_mode": "identity"}))
    add(VariantSpec("C3 Z-score normalization", "C", "Use z-score instead of median/MAD normalization.", "C3", variant_meta={"normalization_mode": "zscore"}))
    add(VariantSpec("C4 Raw credit for all controls", "C", "Alias of no normalization: raw credit drives controls directly.", "C2", variant_meta={"normalization_mode": "identity"}))
    add(VariantSpec("C5 Separate heuristics instead of one shared credit", "C", "Gate uses credit, server step uses inverse cost, prox uses inverse depth.", "C5", variant_meta={"step_mode": "inverse_cost", "prox_mode": "inverse_depth"}))

    # D: Tri-control
    add(VariantSpec("D1 No selective gate", "D", "Always communicate all layers.", "D1", variant_meta={"force_all_layers": True}, ablation_overrides={"use_train_gate": False}))
    add(VariantSpec("D2 Credit only for gate", "D", "Gate uses credit; server step and prox are fixed.", "D2", variant_meta={"step_mode": "fixed", "fixed_eta": 1.0, "prox_mode": "fixed", "fixed_mu": 0.0}))
    add(VariantSpec("D3 Credit only for server step", "D", "Server step uses credit; gate is fixed open and prox is fixed.", "D3", variant_meta={"force_all_layers": True, "step_mode": "credit", "prox_mode": "fixed", "fixed_mu": 0.0}, ablation_overrides={"use_train_gate": False}))
    add(VariantSpec("D4 Credit only for proximal strength", "D", "Prox uses credit; gate is fixed open and server step is fixed.", "D4", variant_meta={"force_all_layers": True, "step_mode": "fixed", "fixed_eta": 1.0, "prox_mode": "credit"}, ablation_overrides={"use_train_gate": False}))
    add(VariantSpec("D5 Gate + prox from credit, server step fixed", "D", "Gate and prox use credit; server step is fixed.", "D5", variant_meta={"step_mode": "fixed", "fixed_eta": 1.0, "prox_mode": "credit"}))

    # E: Local training + sparse update
    add(VariantSpec("E1 No transfer LR", "E", "Disable transfer-based layer-wise local learning rate.", "E1", ablation_overrides={"use_transfer_lr": False}))
    add(VariantSpec("E2 No train-gate scaling", "E", "Do not slow or anchor unselected layers locally.", "E2", ablation_overrides={"use_train_gate": False}))
    add(VariantSpec("E3 No credit-weighted sparse aggregation", "E", "Aggregation ignores client-layer credit weights.", "E3", ablation_overrides={"use_credit_weighted_aggregation": False}))
    add(VariantSpec("E4 No server momentum", "E", "Remove sparse server momentum after aggregation.", "E4", config_overrides={"aggregation_momentum": 0.0}))

    return specs


VARIANT_SPECS = build_variant_specs()

CATEGORY_VARIANTS = {
    "A": ["FedMARS", "A1 No reference sketch", "A2 Raw last-round delta (no EMA)", "A3 Random reference direction", "A7 Single batch per mode"],
    "B": ["FedMARS", "B1 Benefit-only credit", "B2 Benefit + risk only", "B3 Benefit + risk + cost only", "B4 Benefit + risk + probe only", "B5 No depth weighting", "B6 Reverse depth weighting", "B7 Raw cost penalty", "B8 No negative-alignment penalty"],
    "C": ["FedMARS", "C1 Plain mean credit aggregation", "C2 No cross-layer normalization", "C3 Z-score normalization", "C4 Raw credit for all controls", "C5 Separate heuristics instead of one shared credit"],
    "D": ["FedMARS", "D1 No selective gate", "D2 Credit only for gate", "D3 Credit only for server step", "D4 Credit only for proximal strength", "D5 Gate + prox from credit, server step fixed"],
    "E": ["FedMARS", "E1 No transfer LR", "E2 No train-gate scaling", "E3 No credit-weighted sparse aggregation", "E4 No server momentum"],
}

CATEGORY_TITLES = {
    "A": "A. Evidence-building ablations",
    "B": "B. Credit-formula ablations",
    "C": "C. Global credit-processing ablations",
    "D": "D. Tri-control ablations",
    "E": "E. Local training and sparse-update ablations",
}



# ============================================================
# Variant execution
# ============================================================
RUN_CACHE = {}
DATA_CACHE = {}


def build_base_config(seed: int) -> FedMARSConfig:
    return FedMARSConfig(
        random_state=int(seed),
        device=DEVICE,
        num_rounds=int(NUM_ROUNDS),
        warmup_rounds=3,
        client_fraction=CLIENT_FRACTION,
        min_clients_per_round=int(DEFAULT_NUM_CLIENTS),
        local_epochs=int(ABLATION_LOCAL_EPOCHS),
        local_batch_size=int(LOCAL_BATCH_SIZE),
        max_grad_norm=MAX_GRAD_NORM,
        num_clusters=3,
        num_batches_per_cluster=3,
        transfer_probe_batches=3,
        lambda_r=0.35,
        lambda_c=0.02,
        lambda_v=0.30,
        default_budget_fraction=float(COMM_BUDGET_FRACTION),
        default_threshold=float(COMM_THRESHOLD),
        track_server_to_client_bits=TRACK_SERVER_TO_CLIENT_BITS,
        param_bits=int(PARAM_BITS),
        label_smoothing=float(LABEL_SMOOTHING),
    )


def apply_variant_to_config(config: FedMARSConfig, spec: VariantSpec):
    for key, value in spec.config_overrides.items():
        setattr(config, key, value)
    for key, value in spec.ablation_overrides.items():
        setattr(config.ablations, key, value)
    return config


def get_prepared_seed_bundle(seed: int):
    if seed in DATA_CACHE:
        return DATA_CACHE[seed]
    prepared = prepare_dataset_for_ablation(seed=int(seed))
    model_fn = lambda inp=prepared["input_dim"], cls=prepared["num_classes"]: build_model(inp, cls)
    model_bits = count_model_bits(model_fn(), PARAM_BITS)
    val_loader = DataLoader(prepared["val_dataset"], batch_size=SERVER_EVAL_BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(prepared["test_dataset"], batch_size=SERVER_EVAL_BATCH_SIZE, shuffle=False)
    train_clients = fixed_dirichlet_partition(prepared["train_dataset"], seed=int(seed), alpha=float(ABLATION_ALPHA))
    init_model = model_fn().to(DEVICE)
    init_state = detach_state_dict(init_model)
    bundle = {
        "prepared": prepared,
        "model_fn": model_fn,
        "model_bits": model_bits,
        "val_loader": val_loader,
        "test_loader": test_loader,
        "train_clients": train_clients,
        "init_state": init_state,
    }
    DATA_CACHE[seed] = bundle
    return bundle


def run_single_variant(spec: VariantSpec):
    if spec.cache_key in RUN_CACHE:
        cached = RUN_CACHE[spec.cache_key].copy()
        cached["method"] = spec.name
        return cached

    run_rows = []
    for seed in EXPERIMENT_SEEDS:
        set_seed(int(seed))
        bundle = get_prepared_seed_bundle(int(seed))
        config = apply_variant_to_config(build_base_config(seed=int(seed)), spec)

        with ExitStack() as stack:
            if spec.patch_factory is not None:
                for patch_obj in spec.patch_factory():
                    stack.enter_context(patch_obj)

            trainer = HookedFedMARS(
                model=bundle["model_fn"](),
                config=config,
                criterion=nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING),
                variant_meta=copy.deepcopy(spec.variant_meta),
            )
            if spec.trainer_post_init is not None:
                spec.trainer_post_init(trainer)

            load_state_dict_(trainer.model, bundle["init_state"])
            history = trainer.fit(
                bundle["train_clients"],
                server_val_loader=bundle["val_loader"],
                server_test_loader=None,
            )
            final_test_metrics = trainer.evaluate(bundle["test_loader"])

        run_rows.append({
            "method": spec.name,
            "seed": int(seed),
            "final_test_accuracy": float(final_test_metrics["accuracy"]),
            "final_test_loss": float(final_test_metrics["loss"]),
            "best_val_accuracy": float(best_metric_from_history(history, "accuracy")),
            "bits_to_target": float(bits_to_target_from_history(history)),
            "rounds_to_target": float(rounds_to_target_from_history(history)),
            "total_bits": float(total_bits_from_history(history)),
            "mean_selected_layer_ratio": float(selected_ratio_from_history(history)),
            "mean_drift": float(mean_drift_from_history(history)),
        })

    run_df = pd.DataFrame(run_rows)
    summary = pd.DataFrame([{
        "method": spec.name,
        "final_test_accuracy_mean": float(run_df["final_test_accuracy"].mean()),
        "final_test_accuracy_std": float(run_df["final_test_accuracy"].std(ddof=0)),
        "best_val_accuracy_mean": float(run_df["best_val_accuracy"].mean()),
        "bits_to_target_mean": float(run_df["bits_to_target"].dropna().mean()) if run_df["bits_to_target"].notna().any() else np.nan,
        "rounds_to_target_mean": float(run_df["rounds_to_target"].dropna().mean()) if run_df["rounds_to_target"].notna().any() else np.nan,
        "total_bits_mean": float(run_df["total_bits"].mean()),
        "mean_selected_layer_ratio": float(run_df["mean_selected_layer_ratio"].mean()),
        "mean_drift": float(run_df["mean_drift"].mean()),
    }])

    result = {
        "method": spec.name,
        "run_df": run_df,
        "summary_df": summary,
    }
    RUN_CACHE[spec.cache_key] = copy.deepcopy(result)
    return result


def run_category(category_key: str):
    names = CATEGORY_VARIANTS[category_key]
    rows = []
    raw_rows = []
    for name in names:
        spec = VARIANT_SPECS[name]
        result = run_single_variant(spec)
        rows.append(result["summary_df"])
        raw_rows.append(result["run_df"])
    summary_df = pd.concat(rows, ignore_index=True)
    raw_df = pd.concat(raw_rows, ignore_index=True)

    fedmars_row = summary_df.loc[summary_df["method"] == "FedMARS"]
    if not fedmars_row.empty:
        fedmars_row = fedmars_row.iloc[0]
        summary_df["delta_vs_fedmars"] = summary_df["final_test_accuracy_mean"] - float(fedmars_row["final_test_accuracy_mean"])
        summary_df["delta_bits_vs_fedmars"] = summary_df["total_bits_mean"] - float(fedmars_row["total_bits_mean"])
    else:
        summary_df["delta_vs_fedmars"] = np.nan
        summary_df["delta_bits_vs_fedmars"] = np.nan

    variant_map = pd.DataFrame([
        {
            "method": VARIANT_SPECS[name].name,
            "what_changed": VARIANT_SPECS[name].description,
        }
        for name in names
    ])

    summary_df = summary_df.sort_values(
        ["final_test_accuracy_mean", "best_val_accuracy_mean", "method"],
        ascending=[False, False, True]
    ).reset_index(drop=True)

    return variant_map, summary_df, raw_df


def format_summary_table(summary_df: pd.DataFrame):
    df = summary_df.copy()
    return (
        df.style
        .hide(axis="index")
        .format({
            "final_test_accuracy_mean": "{:.4f}",
            "final_test_accuracy_std": "{:.4f}",
            "best_val_accuracy_mean": "{:.4f}",
            "bits_to_target_mean": "{:,.0f}",
            "rounds_to_target_mean": "{:,.1f}",
            "total_bits_mean": "{:,.0f}",
            "mean_selected_layer_ratio": "{:.4f}",
            "mean_drift": "{:.6f}",
            "delta_vs_fedmars": "{:+.4f}",
            "delta_bits_vs_fedmars": "{:+,.0f}",
        })
        .apply(lambda row: ["font-weight:700;background-color:#eef6ff" if row["method"] == "FedMARS" else "" for _ in row], axis=1)
    )


def format_variant_map(variant_map: pd.DataFrame):
    return (
        variant_map.style
        .hide(axis="index")
        .apply(lambda row: ["font-weight:700;background-color:#eef6ff" if row["method"] == "FedMARS" else "" for _ in row], axis=1)
    )


def display_category(category_key: str):
    variant_map, summary_df, raw_df = run_category(category_key)

    prepared = prepare_dataset_for_ablation(seed=int(EXPERIMENT_SEEDS[0]))
    protocol_df = pd.DataFrame([{
        "dataset": prepared["dataset_name"],
        "dataset_source": prepared["source"],
        "rows": prepared["rows"],
        "features": prepared["features"],
        "classes": prepared["classes"],
        "alpha": ABLATION_ALPHA,
        "local_epochs": ABLATION_LOCAL_EPOCHS,
        "rounds": NUM_ROUNDS,
        "seeds": len(EXPERIMENT_SEEDS),
        "clients": DEFAULT_NUM_CLIENTS,
        "batch_size": LOCAL_BATCH_SIZE,
        "budget_fraction": COMM_BUDGET_FRACTION,
        "threshold": COMM_THRESHOLD,
        "device": DEVICE,
    }])

    display(Markdown(f"## {CATEGORY_TITLES[category_key]}"))
    display(Markdown("### Protocol"))
    display(protocol_df)
    display(Markdown("### Variant map"))
    display(format_variant_map(variant_map))
    display(Markdown("### Summary table"))
    display(format_summary_table(summary_df))
    display(Markdown("### Per-seed raw results"))
    display(
        raw_df.sort_values(["method", "seed"]).reset_index(drop=True).style.hide(axis="index").format({
            "final_test_accuracy": "{:.4f}",
            "final_test_loss": "{:.4f}",
            "best_val_accuracy": "{:.4f}",
            "bits_to_target": "{:,.0f}",
            "rounds_to_target": "{:,.1f}",
            "total_bits": "{:,.0f}",
            "mean_selected_layer_ratio": "{:.4f}",
            "mean_drift": "{:.6f}",
        })
    )


display(Markdown("## Setup check"))
display(pd.DataFrame([{
    "dataset": fetch_iris_source()["dataset_name"],
    "source": fetch_iris_source()["source"],
    "rounds": NUM_ROUNDS,
    "seeds": len(EXPERIMENT_SEEDS),
    "clients": DEFAULT_NUM_CLIENTS,
    "alpha": ABLATION_ALPHA,
    "local_epochs": ABLATION_LOCAL_EPOCHS,
    "batch_size": LOCAL_BATCH_SIZE,
    "budget_fraction": COMM_BUDGET_FRACTION,
    "threshold": COMM_THRESHOLD,
    "device": DEVICE,
    "note": "Rebuilt to match the stronger earlier validation-style setup, not the simplified demo harness.",
}]))


## Setup check

,dataset,source,rounds,seeds,clients,alpha,local_epochs,batch_size,budget_fraction,threshold,device,note
0,Iris (UCI id=53),ucimlrepo,40,5,10,0.5,5,32,0.35,-0.05,cuda,Rebuilt to match the stronger earlier validati...


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [2]:
display_category('A')

## A. Evidence-building ablations

### Protocol

,dataset,dataset_source,rows,features,classes,alpha,local_epochs,rounds,seeds,clients,batch_size,budget_fraction,threshold,device
0,Iris (UCI id=53),ucimlrepo,150,4,3,0.5,5,40,5,10,32,0.35,-0.05,cuda


### Variant map

method,what_changed
FedMARS,Full FedMARS with the stronger validation-style configuration.
A1 No reference sketch,Remove the global directional reference.
A2 Raw last-round delta (no EMA),Use raw unit delta instead of EMA reference memory.
A3 Random reference direction,Replace the reference sketch with a random unit direction.
A7 Single batch per mode,One batch per mode instead of repeated mini-batch averaging.


### Summary table

method,final_test_accuracy_mean,final_test_accuracy_std,best_val_accuracy_mean,bits_to_target_mean,rounds_to_target_mean,total_bits_mean,mean_selected_layer_ratio,mean_drift,delta_vs_fedmars,delta_bits_vs_fedmars
FedMARS,0.9867,0.0163,0.9867,"12,081,408",10.8,"39,347,200",0.6917,0.000148,+0.0000,+0
A2 Raw last-round delta (no EMA),0.9800,0.0163,0.9867,"13,201,920",12.0,"39,347,200",0.6917,0.000150,-0.0067,+0
A7 Single batch per mode,0.9800,0.0163,0.9867,"11,334,400",10.0,"39,347,200",0.6917,0.000148,-0.0067,+0
A1 No reference sketch,0.9733,0.0249,0.9867,"15,069,440",14.0,"39,347,200",0.6917,0.000142,-0.0133,+0
A3 Random reference direction,0.9733,0.0249,0.9867,"16,376,704",15.4,"39,347,200",0.6917,0.000161,-0.0133,+0


### Per-seed raw results

method,seed,final_test_accuracy,final_test_loss,best_val_accuracy,bits_to_target,rounds_to_target,total_bits,mean_selected_layer_ratio,mean_drift
A1 No reference sketch,2026,0.9667,0.2029,1.0000,"15,069,440",14.0,"39,347,200",0.6917,0.000117
A1 No reference sketch,2027,0.9667,0.1601,0.9667,"16,003,200",15.0,"39,347,200",0.6917,0.000140
A1 No reference sketch,2028,0.9333,0.2399,1.0000,"10,400,640",9.0,"39,347,200",0.6917,0.000132
A1 No reference sketch,2029,1.0000,0.1427,1.0000,"18,804,480",18.0,"39,347,200",0.6917,0.000153
A1 No reference sketch,2030,1.0000,0.1161,0.9667,"15,069,440",14.0,"39,347,200",0.6917,0.000168
A2 Raw last-round delta (no EMA),2026,0.9667,0.1586,1.0000,"10,400,640",9.0,"39,347,200",0.6917,0.000141
A2 Raw last-round delta (no EMA),2027,0.9667,0.1483,0.9667,"14,135,680",13.0,"39,347,200",0.6917,0.000153
A2 Raw last-round delta (no EMA),2028,1.0000,0.2154,1.0000,"15,069,440",14.0,"39,347,200",0.6917,0.000125
A2 Raw last-round delta (no EMA),2029,1.0000,0.1205,1.0000,"11,334,400",10.0,"39,347,200",0.6917,0.000141
A2 Raw last-round delta (no EMA),2030,0.9667,0.0936,0.9667,"15,069,440",14.0,"39,347,200",0.6917,0.000191


In [3]:
display_category('B')

## B. Credit-formula ablations

### Protocol

,dataset,dataset_source,rows,features,classes,alpha,local_epochs,rounds,seeds,clients,batch_size,budget_fraction,threshold,device
0,Iris (UCI id=53),ucimlrepo,150,4,3,0.5,5,40,5,10,32,0.35,-0.05,cuda


### Variant map

method,what_changed
FedMARS,Full FedMARS with the stronger validation-style configuration.
B1 Benefit-only credit,Keep only the benefit term from the layer score.
B2 Benefit + risk only,"Remove cost, probe, and depth terms."
B3 Benefit + risk + cost only,Remove probe and depth terms.
B4 Benefit + risk + probe only,Remove cost and depth terms.
B5 No depth weighting,Make all layers share the same depth weight.
B6 Reverse depth weighting,Reverse the depth weighting across layers.
B7 Raw cost penalty,Use raw layer cost instead of sqrt(cost).
B8 No negative-alignment penalty,Remove the direct negative-alignment penalty inside risk.


### Summary table

method,final_test_accuracy_mean,final_test_accuracy_std,best_val_accuracy_mean,bits_to_target_mean,rounds_to_target_mean,total_bits_mean,mean_selected_layer_ratio,mean_drift,delta_vs_fedmars,delta_bits_vs_fedmars
FedMARS,0.9867,0.0163,0.9867,"12,081,408",10.8,"39,347,200",0.6917,0.000148,+0.0000,+0
B1 Benefit-only credit,0.9800,0.0163,0.9867,"15,816,448",14.8,"39,347,200",0.6917,0.000115,-0.0067,+0
B2 Benefit + risk only,0.9800,0.0163,0.9867,"12,692,864",11.4,"39,357,440",0.6650,0.000121,-0.0067,"+10,240"
B3 Benefit + risk + cost only,0.9800,0.0163,0.9867,"16,053,888",15.8,"38,609,920",0.6317,0.000133,-0.0067,"-737,280"
B4 Benefit + risk + probe only,0.9800,0.0163,0.9867,"15,069,440",14.0,"39,347,200",0.6917,0.000136,-0.0067,+0
B5 No depth weighting,0.9800,0.0163,0.9867,"12,828,416",11.6,"39,347,200",0.6917,0.000139,-0.0067,+0
B6 Reverse depth weighting,0.9800,0.0163,0.9867,"12,454,912",11.2,"39,347,200",0.6917,0.000114,-0.0067,+0
B7 Raw cost penalty,0.9800,0.0163,0.9867,"11,521,152",10.2,"39,347,200",0.6917,0.000148,-0.0067,+0
B8 No negative-alignment penalty,0.9800,0.0163,0.9867,"16,936,960",16.0,"39,347,200",0.6917,0.000147,-0.0067,+0


### Per-seed raw results

method,seed,final_test_accuracy,final_test_loss,best_val_accuracy,bits_to_target,rounds_to_target,total_bits,mean_selected_layer_ratio,mean_drift
B1 Benefit-only credit,2026,0.9667,0.1658,1.0000,"11,334,400",10.0,"39,347,200",0.6917,0.000113
B1 Benefit-only credit,2027,0.9667,0.1469,0.9667,"24,407,040",24.0,"39,347,200",0.6917,0.000116
B1 Benefit-only credit,2028,1.0000,0.2256,1.0000,"16,936,960",16.0,"39,347,200",0.6917,0.000091
B1 Benefit-only credit,2029,1.0000,0.1344,1.0000,"12,268,160",11.0,"39,347,200",0.6917,0.000107
B1 Benefit-only credit,2030,0.9667,0.1020,0.9667,"14,135,680",13.0,"39,347,200",0.6917,0.000147
B2 Benefit + risk only,2026,0.9667,0.1773,1.0000,"11,858,560",11.0,"38,937,600",0.6583,0.000135
B2 Benefit + risk only,2027,0.9667,0.1575,0.9667,"16,003,200",15.0,"39,347,200",0.6917,0.000116
B2 Benefit + risk only,2028,1.0000,0.1443,1.0000,"9,288,960",6.0,"40,934,400",0.6833,0.000088
B2 Benefit + risk only,2029,1.0000,0.1443,1.0000,"11,232,000",10.0,"39,244,800",0.6833,0.000115
B2 Benefit + risk only,2030,0.9667,0.1284,0.9667,"15,081,600",15.0,"38,323,200",0.6083,0.000154


In [4]:
display_category('C')

## C. Global credit-processing ablations

### Protocol

,dataset,dataset_source,rows,features,classes,alpha,local_epochs,rounds,seeds,clients,batch_size,budget_fraction,threshold,device
0,Iris (UCI id=53),ucimlrepo,150,4,3,0.5,5,40,5,10,32,0.35,-0.05,cuda


### Variant map

method,what_changed
FedMARS,Full FedMARS with the stronger validation-style configuration.
C1 Plain mean credit aggregation,Use plain mean across client credits.
C2 No cross-layer normalization,Use raw credit directly for controls.
C3 Z-score normalization,Use z-score instead of median/MAD normalization.
C4 Raw credit for all controls,Alias of no normalization: raw credit drives controls directly.
C5 Separate heuristics instead of one shared credit,"Gate uses credit, server step uses inverse cost, prox uses inverse depth."


### Summary table

method,final_test_accuracy_mean,final_test_accuracy_std,best_val_accuracy_mean,bits_to_target_mean,rounds_to_target_mean,total_bits_mean,mean_selected_layer_ratio,mean_drift,delta_vs_fedmars,delta_bits_vs_fedmars
FedMARS,0.9867,0.0163,0.9867,"12,081,408",10.8,"39,347,200",0.6917,0.000148,+0.0000,+0
C1 Plain mean credit aggregation,0.9800,0.0163,0.9867,"12,268,160",11.0,"39,347,200",0.6917,0.000146,-0.0067,+0
C2 No cross-layer normalization,0.9800,0.0163,0.9867,"15,069,440",14.0,"39,347,200",0.6917,0.000121,-0.0067,+0
C2 No cross-layer normalization,0.9800,0.0163,0.9867,"15,069,440",14.0,"39,347,200",0.6917,0.000121,-0.0067,+0
C3 Z-score normalization,0.9800,0.0163,0.9867,"13,762,176",12.6,"39,347,200",0.6917,0.000147,-0.0067,+0
C5 Separate heuristics instead of one shared credit,0.9800,0.0163,0.9867,"14,135,680",13.0,"39,347,200",0.6917,0.000151,-0.0067,+0


### Per-seed raw results

method,seed,final_test_accuracy,final_test_loss,best_val_accuracy,bits_to_target,rounds_to_target,total_bits,mean_selected_layer_ratio,mean_drift
C1 Plain mean credit aggregation,2026,0.9667,0.1583,1.0000,"10,400,640",9.0,"39,347,200",0.6917,0.000143
C1 Plain mean credit aggregation,2027,0.9667,0.1450,0.9667,"14,135,680",13.0,"39,347,200",0.6917,0.000153
C1 Plain mean credit aggregation,2028,0.9667,0.2516,1.0000,"17,870,720",17.0,"39,347,200",0.6917,0.000107
C1 Plain mean credit aggregation,2029,1.0000,0.1402,1.0000,"4,798,080",3.0,"39,347,200",0.6917,0.000135
C1 Plain mean credit aggregation,2030,1.0000,0.0937,0.9667,"14,135,680",13.0,"39,347,200",0.6917,0.000191
C2 No cross-layer normalization,2026,0.9667,0.1609,1.0000,"10,400,640",9.0,"39,347,200",0.6917,0.000114
C2 No cross-layer normalization,2026,0.9667,0.1609,1.0000,"10,400,640",9.0,"39,347,200",0.6917,0.000114
C2 No cross-layer normalization,2027,0.9667,0.1487,0.9667,"17,870,720",17.0,"39,347,200",0.6917,0.000121
C2 No cross-layer normalization,2027,0.9667,0.1487,0.9667,"17,870,720",17.0,"39,347,200",0.6917,0.000121
C2 No cross-layer normalization,2028,1.0000,0.2125,1.0000,"14,135,680",13.0,"39,347,200",0.6917,0.000101


In [5]:
display_category('D')

## D. Tri-control ablations

### Protocol

,dataset,dataset_source,rows,features,classes,alpha,local_epochs,rounds,seeds,clients,batch_size,budget_fraction,threshold,device
0,Iris (UCI id=53),ucimlrepo,150,4,3,0.5,5,40,5,10,32,0.35,-0.05,cuda


### Variant map

method,what_changed
FedMARS,Full FedMARS with the stronger validation-style configuration.
D1 No selective gate,Always communicate all layers.
D2 Credit only for gate,Gate uses credit; server step and prox are fixed.
D3 Credit only for server step,Server step uses credit; gate is fixed open and prox is fixed.
D4 Credit only for proximal strength,Prox uses credit; gate is fixed open and server step is fixed.
"D5 Gate + prox from credit, server step fixed",Gate and prox use credit; server step is fixed.


### Summary table

method,final_test_accuracy_mean,final_test_accuracy_std,best_val_accuracy_mean,bits_to_target_mean,rounds_to_target_mean,total_bits_mean,mean_selected_layer_ratio,mean_drift,delta_vs_fedmars,delta_bits_vs_fedmars
FedMARS,0.9867,0.0163,0.9867,"12,081,408",10.8,"39,347,200",0.6917,0.000148,+0.0000,+0
D2 Credit only for gate,0.9800,0.0163,0.9867,"11,707,904",10.4,"39,347,200",0.6917,0.000155,-0.0067,+0
"D5 Gate + prox from credit, server step fixed",0.9733,0.0249,0.9867,"13,015,168",11.8,"39,347,200",0.6917,0.000155,-0.0133,+0
D4 Credit only for proximal strength,0.9733,0.0249,0.9867,"12,155,136",7.6,"63,974,400",1.0000,0.000107,-0.0133,"+24,627,200"
D3 Credit only for server step,0.9600,0.0249,0.9867,"12,475,008",7.8,"63,974,400",1.0000,0.000099,-0.0267,"+24,627,200"
D1 No selective gate,0.9400,0.0133,0.9867,"12,475,008",7.8,"63,974,400",1.0000,0.000098,-0.0467,"+24,627,200"


### Per-seed raw results

method,seed,final_test_accuracy,final_test_loss,best_val_accuracy,bits_to_target,rounds_to_target,total_bits,mean_selected_layer_ratio,mean_drift
D1 No selective gate,2026,0.9333,0.0866,1.0000,"12,794,880",8.0,"63,974,400",1.0000,0.000093
D1 No selective gate,2027,0.9667,0.1933,0.9667,"11,195,520",7.0,"63,974,400",1.0000,0.000098
D1 No selective gate,2028,0.9333,0.0967,1.0000,"20,791,680",13.0,"63,974,400",1.0000,0.000076
D1 No selective gate,2029,0.9333,0.0613,1.0000,"4,798,080",3.0,"63,974,400",1.0000,0.000098
D1 No selective gate,2030,0.9333,0.1079,0.9667,"12,794,880",8.0,"63,974,400",1.0000,0.000127
D2 Credit only for gate,2026,0.9667,0.1237,1.0000,"10,400,640",9.0,"39,347,200",0.6917,0.000147
D2 Credit only for gate,2027,0.9667,0.1221,0.9667,"12,268,160",11.0,"39,347,200",0.6917,0.000153
D2 Credit only for gate,2028,1.0000,0.1584,1.0000,"12,268,160",11.0,"39,347,200",0.6917,0.000126
D2 Credit only for gate,2029,1.0000,0.0865,1.0000,"11,334,400",10.0,"39,347,200",0.6917,0.000151
D2 Credit only for gate,2030,0.9667,0.0842,0.9667,"12,268,160",11.0,"39,347,200",0.6917,0.000199


In [6]:
display_category('E')

## E. Local training and sparse-update ablations

### Protocol

,dataset,dataset_source,rows,features,classes,alpha,local_epochs,rounds,seeds,clients,batch_size,budget_fraction,threshold,device
0,Iris (UCI id=53),ucimlrepo,150,4,3,0.5,5,40,5,10,32,0.35,-0.05,cuda


### Variant map

method,what_changed
FedMARS,Full FedMARS with the stronger validation-style configuration.
E1 No transfer LR,Disable transfer-based layer-wise local learning rate.
E2 No train-gate scaling,Do not slow or anchor unselected layers locally.
E3 No credit-weighted sparse aggregation,Aggregation ignores client-layer credit weights.
E4 No server momentum,Remove sparse server momentum after aggregation.


### Summary table

method,final_test_accuracy_mean,final_test_accuracy_std,best_val_accuracy_mean,bits_to_target_mean,rounds_to_target_mean,total_bits_mean,mean_selected_layer_ratio,mean_drift,delta_vs_fedmars,delta_bits_vs_fedmars
FedMARS,0.9867,0.0163,0.9867,"12,081,408",10.8,"39,347,200",0.6917,0.000148,+0.0000,+0
E1 No transfer LR,0.9800,0.0163,0.9867,"13,948,928",12.8,"39,347,200",0.6917,0.000156,-0.0067,+0
E2 No train-gate scaling,0.9800,0.0163,0.9867,"12,268,160",11.0,"39,347,200",0.6917,0.000125,-0.0067,+0
E3 No credit-weighted sparse aggregation,0.9600,0.0327,0.9867,"16,003,200",15.0,"39,347,200",0.6917,0.000155,-0.0267,+0
E4 No server momentum,0.8133,0.1002,0.9067,"26,274,560",26.0,"39,347,200",0.6917,0.000010,-0.1733,+0


### Per-seed raw results

method,seed,final_test_accuracy,final_test_loss,best_val_accuracy,bits_to_target,rounds_to_target,total_bits,mean_selected_layer_ratio,mean_drift
E1 No transfer LR,2026,0.9667,0.1401,1.0000,"10,400,640",9.0,"39,347,200",0.6917,0.000153
E1 No transfer LR,2027,0.9667,0.1383,0.9667,"13,201,920",12.0,"39,347,200",0.6917,0.000159
E1 No transfer LR,2028,1.0000,0.2302,1.0000,"16,936,960",16.0,"39,347,200",0.6917,0.000124
E1 No transfer LR,2029,1.0000,0.1311,1.0000,"13,201,920",12.0,"39,347,200",0.6917,0.000141
E1 No transfer LR,2030,0.9667,0.0880,0.9667,"16,003,200",15.0,"39,347,200",0.6917,0.000203
E2 No train-gate scaling,2026,0.9667,0.1936,1.0000,"20,672,000",20.0,"39,347,200",0.6917,0.000125
E2 No train-gate scaling,2027,0.9667,0.1619,0.9667,"13,201,920",12.0,"39,347,200",0.6917,0.000130
E2 No train-gate scaling,2028,1.0000,0.2728,1.0000,"15,069,440",14.0,"39,347,200",0.6917,0.000098
E2 No train-gate scaling,2029,1.0000,0.2011,1.0000,"4,798,080",3.0,"39,347,200",0.6917,0.000109
E2 No train-gate scaling,2030,0.9667,0.1122,0.9667,"7,599,360",6.0,"39,347,200",0.6917,0.000165
